# Cointegration and error correction: lab notebook

**Statistical Modelling and Forecasting, ARIMA and vector autoregression**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen

rng = np.random.default_rng(6)
n = 300

### 1. A cointegrated pair: y = 10 + 1.5 x + u, with u an AR(1) (rho = 0.7) and x a random walk

In [ ]:
# 1. A cointegrated pair: y = 10 + 1.5 x + u, with u an AR(1) (rho = 0.7) and x a random walk
x = np.cumsum(rng.standard_normal(n))
u = np.zeros(n)
for t in range(1, n):
    u[t] = 0.7 * u[t - 1] + rng.standard_normal()
y = 10 + 1.5 * x + u
print("ADF p-value of x, y (levels):", round(adfuller(x)[1], 3), round(adfuller(y)[1], 3), "| of their differences:", round(adfuller(np.diff(x))[1], 4), round(adfuller(np.diff(y))[1], 4))

### 2. Engle-Granger: long-run regression, then a unit-root test on the residuals

In [ ]:
# 2. Engle-Granger: long-run regression, then a unit-root test on the residuals
ols = sm.OLS(y, sm.add_constant(x)).fit()
print("long-run beta:", round(ols.params[1], 3), "(true 1.5)")
resid = ols.resid
tau = adfuller(resid, maxlag=1, regression="n", autolag=None)[0]
print("ADF tau on the residuals:", round(tau, 2), "-> compare with the Engle-Granger value -3.34, not -1.95")
t, p, crit = coint(y, x, trend="c", autolag=None, maxlag=1)
print("statsmodels coint: tau =", round(t, 2), "p =", round(p, 4), "critical values 1/5/10%:", np.round(crit, 2))

### 3. Error-correction model: dy_t = gamma dx_t + alpha z_{t-1} + e_t

In [ ]:
# 3. Error-correction model: dy_t = gamma dx_t + alpha z_{t-1} + e_t
dy, dx = np.diff(y), np.diff(x)
Z = np.column_stack([np.ones(n - 1), dx, resid[:-1]])
ecm = sm.OLS(dy, Z).fit()
alpha = ecm.params[2]
print("gamma:", round(ecm.params[1], 3), "| alpha:", round(alpha, 3), "(true -0.3) | t(alpha) =", round(ecm.tvalues[2], 1))
print("half-life of a deviation:", round(np.log(0.5) / np.log(1 + alpha), 2), "periods")

### 4. Two independent random walks: spurious regression and the test

In [ ]:
# 4. Two independent random walks: spurious regression and the test
a = np.cumsum(rng.standard_normal(n)); b = np.cumsum(rng.standard_normal(n))
sp = sm.OLS(b, sm.add_constant(a)).fit()
print("independent walks: R2 =", round(sp.rsquared, 2), "| t(beta) =", round(sp.tvalues[1], 1), "| EG p-value:", round(coint(b, a)[1], 3))
rej = sum(coint(np.cumsum(rng.standard_normal(150)), np.cumsum(rng.standard_normal(150)))[1] < 0.05 for _ in range(200))
print("EG rejections among 200 independent pairs:", rej, "(about 5% expected)")

### 5. Johansen test for the cointegration rank (constant, one lagged difference)

In [ ]:
# 5. Johansen test for the cointegration rank (constant, one lagged difference)
J = coint_johansen(np.column_stack([y, x]), det_order=0, k_ar_diff=1)
print("trace statistics:", np.round(J.lr1, 1), "| 5% critical values:", np.round(J.cvt[:, 1], 1))
print("rank:", int(np.sum(J.lr1 > J.cvt[:, 1])))

### 6. The numerical problem

In [ ]:
# 6. The numerical problem
be, x0, y0, al, ga, dx0 = 1.5, 30, 50, -0.25, 0.5, 2
z = y0 - be * x0
print("z =", z, "| dy =", ga * dx0 + al * z, "| half-life:", round(np.log(0.5) / np.log(1 + al), 2))

## Expected output

Running all cells above should print:

```
ADF p-value of x, y (levels): 0.518 0.342 | of their differences: 0.0 0.0
long-run beta: 1.485 (true 1.5)
ADF tau on the residuals: -7.61 -> compare with the Engle-Granger value -3.34, not -1.95
statsmodels coint: tau = -7.61 p = 0.0 critical values 1/5/10%: [-3.93 -3.36 -3.06]
gamma: 1.551 | alpha: -0.304 (true -0.3) | t(alpha) = -7.3
half-life of a deviation: 1.92 periods
independent walks: R2 = 0.18 | t(beta) = 8.0 | EG p-value: 0.458
EG rejections among 200 independent pairs: 15 (about 5% expected)
trace statistics: [55.8  2.4] | 5% critical values: [15.5  3.8]
rank: 1
z = 5.0 | dy = -0.25 | half-life: 2.41
```

## Practice exercises

Try these in the cells below.

1. Set cointegration to "on" and run the test with the default settings. Read the ADF τ of the residuals.
2. Switch to "independent random walks" and press New sample many times. How often does the test reject?
3. Make the residual AR(1) parameter 0.95 (slow adjustment). What happens to α and to the power of the test?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
